In [2]:
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

In [3]:
# Toy data: two interleaving half-circles (not linearly separable)
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [4]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)


In [5]:
# Small MLP: 2 -> 16 -> 16 -> 1
model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1)
)

In [6]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [7]:
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")


epoch   0  loss 0.7235
epoch  50  loss 0.2202
epoch 100  loss 0.0866
epoch 150  loss 0.0672
epoch 200  loss 0.0647
epoch 250  loss 0.0617


In [8]:
model.eval()
with torch.no_grad():
    preds = (torch.sigmoid(model(X_test)) > 0.5).float()
    print(f"test accuracy: {(preds == y_test).float().mean().item():.3f}")

test accuracy: 0.990
